# Colab Worker TTS

Notebook này clone repo từ GitHub và chạy `colab/worker.py`.
Chỉnh `SERVER_URL` và `EMAIL` ở cell 2 rồi chạy cả hai cell.


In [ ]:
#@title 1. Cài đặt dependencies & đồng bộ worker.py từ GitHub
import os
import subprocess
import sys
import time

import torch

GITHUB_USER = ""  #@param {type:"string"}
GITHUB_REPO = ""  #@param {type:"string"}
BRANCH = "main"               #@param {type:"string"}

if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ KHÔNG có GPU! Worker sẽ chạy rất chậm trên CPU.")

repo_dir = f"/content/{GITHUB_REPO}"
repo_url = f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
if os.path.exists(repo_dir):
    print(f"🔄 Pull latest: {repo_dir}")
    subprocess.run(["git", "-C", repo_dir, "fetch", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", repo_dir, "checkout", BRANCH], check=False)
    subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only", "origin", BRANCH], check=False)
else:
    print(f"⬇️ Clone repo: {repo_url}")
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, repo_dir], check=False)

print("📦 Đang cài đặt dependencies...")
for _ in range(3):
    ret = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "soundfile", "pydub", "soxr", "websockets", "httpx"], check=False)
    if ret.returncode == 0:
        break
    time.sleep(2)
for _ in range(3):
    ret = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "omnivoice", "--no-deps"], check=False)
    if ret.returncode == 0:
        break
    time.sleep(2)

sys.path.append(os.path.join(repo_dir, "colab"))
print("✅ Đồng bộ và cài đặt hoàn tất.")


In [ ]:
#@title 2. Chạy WebSocket Worker
import os
import subprocess
import sys

SERVER_URL = ""  #@param {type:"string"}
EMAIL = ""       #@param {type:"string"}
WORKER_SESSION_ID = ""       #@param {type:"string"}

if not SERVER_URL.strip():
    raise ValueError("SERVER_URL không được để trống. Hãy nhập URL Cloudflare/server.")
if not EMAIL.strip():
    raise ValueError("EMAIL không được để trống.")

worker_path = os.path.join(repo_dir, "colab", "worker.py")
print(f"🚀 Running worker for {EMAIL}...")
subprocess.run([
    sys.executable,
    worker_path,
    "--server-url", SERVER_URL.strip(),
    "--email", EMAIL.strip(),
    "--worker-session-id", WORKER_SESSION_ID.strip(),
], check=False)
